# RNA Structure Processing Tutorial

This notebook demonstrates how to use the RNA structure processing pipeline for various tasks.

## Setup

First, let's import the necessary modules and set up our environment.

In [1]:
import sys
sys.path.append('.')

from utils.pdb_downloader import PDBDownloader
from utils.pdb_to_npy import PDBToNumpyConverter
from utils.npy_to_pdb import NumpyToPDBConverter
from utils.extract_loops import LoopExtractor
from utils.extract_rna_segments import RNAExtractor
from utils.extract_sequence import SequenceExtractor
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

## A. Data Acquisition

### Downloading RNA structures from PDB

In [3]:
# Initialize the downloader
downloader = PDBDownloader(output_dir="downloaded_rna_pdbs2")

# Example 1: Download specific PDB IDs
pdb_ids = ['1H3E','1NBS','1ZDK','2BQ5','2C51','2YU9','3LOB','3NKB','3OWI','3SIU','4B3O','4BW0','4FNJ','4GMA','4TUX','4V2S','4V7L','4V84','4W2H','4X4T','4X4U','5B63','5C0X','5ML7','5U4J','5V3F','5VT0','5WSG','5Y87','6CF2','6KWQ','6O1K','6SPD','6SWD','6T0R','6TQA','6UFK','6UQ1','6XBU','6XH2','6Z8K','6ZJ3','7ANE','7AOI','7D5T','7D7V','7D81','7OG0','7ORI','7PZY','7QCA','7S37','7TQB','7UZ0','7VW3','7XWZ','7XYB','7YFX','8ASW','8C83','8C95','8CGV','8CRX','8CVO','8D4A','8E29','8EG8','8FEO','8FNI','8G4W','8HSG','8ID2','8IFM','8IN8','8JCH','8JY0','8K87','8OIT','8OMR','8OPP','8OUF','8P60','8PIM','8QCQ','8QPP','8R08','8RAM','8RAN','8S52','8SA3','8T5D','8TDZ','8THQ','8TUX','8UTG','8VAJ','8VAT','8VES','8XYC','9BH5','9BZ1','9C3H','9GUT']  # Replace with actual PDB IDs
downloader.download_pdbs(pdb_ids)

# # Example 2: Search and download by criteria
# criteria = {
#     'resolution': 3.0,  # Maximum resolution
#     'rna_only': True,  # RNA-only structures
#     'min_length': 50   # Minimum sequence length
# }
# downloader.search_and_download(criteria)


File already exists: downloaded_rna_pdbs2/1h3e.pdb
File already exists: downloaded_rna_pdbs2/1nbs.pdb
File already exists: downloaded_rna_pdbs2/1zdk.pdb
File already exists: downloaded_rna_pdbs2/2bq5.pdb
File already exists: downloaded_rna_pdbs2/2c51.pdb
File already exists: downloaded_rna_pdbs2/2yu9.pdb
File already exists: downloaded_rna_pdbs2/3lob.pdb
File already exists: downloaded_rna_pdbs2/3nkb.pdb
File already exists: downloaded_rna_pdbs2/3owi.pdb
File already exists: downloaded_rna_pdbs2/3siu.pdb
File already exists: downloaded_rna_pdbs2/4b3o.pdb
File already exists: downloaded_rna_pdbs2/4bw0.pdb
File already exists: downloaded_rna_pdbs2/4fnj.pdb
File already exists: downloaded_rna_pdbs2/4gma.pdb
File already exists: downloaded_rna_pdbs2/4tux.pdb
File already exists: downloaded_rna_pdbs2/4v2s.pdb
Downloaded: downloaded_rna_pdbs2/4v7l.cif
Downloaded: downloaded_rna_pdbs2/4v84.cif
Downloaded: downloaded_rna_pdbs2/4w2h.cif
File already exists: downloaded_rna_pdbs2/4x4t.pdb
File a

In [ ]:
   # Download specific PDB IDs with limit
   !python pdb_downloader.py --pdb-ids 1ABC 2XYZ --max-pdbs 5

   # Download from file with limit
   !python pdb_downloader.py --input-file pdb_list.txt --max-pdbs 10

   # Search and download with limit
   !python pdb_downloader.py --search --rna-only --max-pdbs 20

## B. Structure Processing

### Converting PDB files to NumPy arrays

In [ ]:
# Initialize the converter
converter = PDBToNumpyConverter(
    processed_dir="processed_pdbs",
    npy_dir="npy_files"
)

# Convert all PDB files
converter.convert_all_pdbs()

# Example: Load and visualize a NumPy array
npy_file = "npy_files/1ABC.npy"
if os.path.exists(npy_file):
    data = np.load(npy_file)
    print(f"Array shape: {data.shape}")
    print(f"Number of residues: {data.shape[0]}")
    
    # Plot atom positions for first residue
    plt.figure(figsize=(10, 6))
    plt.scatter(data[0, :, 0], data[0, :, 1])
    plt.title("Atom positions for first residue")
    plt.show()

In [ ]:
!python pdb_to_npy.py --input-dir processed_pdbs --output-dir npy_files

### Reconstructing PDB files from NumPy arrays

In [ ]:
# Initialize the converter
converter = NumpyToPDBConverter(
    npy_dir="npy_files",
    output_dir="reconstructed_pdbs"
)

# Convert all NumPy files
converter.convert_all_npy()

# Example: Convert a specific file
npy_file = "npy_files/1ABC.npy"
fasta_file = "sequences/1ABC.fasta"
if os.path.exists(npy_file) and os.path.exists(fasta_file):
    converter.convert_single_file(npy_file, fasta_file)

In [ ]:
!python scripts/npy_to_pdb.py --npy-dir competition/train/coords --fasta-dir competition/train/seqs --output-dir reconstructed_pdbs

## C. RNA Loop Extraction

### Extracting loop regions

In [4]:
import os
import sys
import logging
from pathlib import Path
from utils.extract_loops import LoopExtractor

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)


# Example: Extract loops with different cutoffs
for cutoff in [8.0, 10.0, 12.0]:
    extractor = LoopExtractor(
        input_dir="reconstructed_pdbs_gt30",
        min_length=3,
        max_length=20,  # or whatever max length you want
        distance_cutoff=cutoff,  # your cutoff value
        atom_type="C4'",  # or "N1" or "N9" if you prefer
        output_dir=f"reconstructed_pdbs_gt30/extracted_loops_{cutoff}"  # or whatever output directory you want
    )
    extractor.process()

2025-06-15 13:56:51,722 - INFO - Found 1456 PDB files to process
2025-06-15 13:56:51,723 - INFO - Processing 9D0J_1_1x.pdb
/Users/xiaojuzhang/opt/anaconda3/lib/python3.9/site-packages/Bio/SeqRecord.py:228: BiopythonDeprecationWarning: Using a string as the sequence is deprecated and will raise a TypeError in future. It has been converted to a Seq object.
  warnings.warn(
2025-06-15 13:56:51,735 - INFO - Processing 4U4O_1_2.pdb
2025-06-15 13:56:51,901 - INFO - Processing 4V5C_1_BA.pdb
2025-06-15 13:56:52,135 - INFO - Processing 5L3P_1_x.pdb
/Users/xiaojuzhang/opt/anaconda3/lib/python3.9/site-packages/Bio/SeqRecord.py:228: BiopythonDeprecationWarning: Using a string as the sequence is deprecated and will raise a TypeError in future. It has been converted to a Seq object.
  warnings.warn(
2025-06-15 13:56:52,145 - INFO - Processing 4V5B_1_BA.pdb
2025-06-15 13:56:52,314 - INFO - Processing 5WIT_1_1A.pdb
/Users/xiaojuzhang/opt/anaconda3/lib/python3.9/site-packages/Bio/SeqRecord.py:228: Biop

In [2]:
# # Initialize the extractor
# extractor = LoopExtractor(
#     input_dir="reconstructed_pdbs",
#     output_dir="extracted_loops",
#     distance_cutoff=10.0  # 10Å cutoff
# )

# # Process all files
# extractor.process()

# Example: Extract loops with different cutoffs
for cutoff in [8.0, 10.0, 12.0]:
    extractor = LoopExtractor(
        input_dir="reconstructed_pdbs",
        output_dir=f"extracted_loops_{cutoff}",
        distance_cutoff=cutoff
    )
    extractor.process()

2025-05-27 18:52:15,973 - INFO - Found 1981 PDB files to process
2025-05-27 18:52:15,974 - INFO - Processing 9D0J_1_1x.pdb
/Users/xiaojuzhang/opt/anaconda3/lib/python3.9/site-packages/Bio/SeqRecord.py:228: BiopythonDeprecationWarning: Using a string as the sequence is deprecated and will raise a TypeError in future. It has been converted to a Seq object.
  warnings.warn(
2025-05-27 18:52:16,055 - INFO - Processing 4U4O_1_2.pdb
2025-05-27 18:52:16,261 - INFO - Processing 4V5C_1_BA.pdb
2025-05-27 18:52:16,535 - INFO - Processing 1HNW_1_X.pdb
/Users/xiaojuzhang/opt/anaconda3/lib/python3.9/site-packages/Bio/SeqRecord.py:228: BiopythonDeprecationWarning: Using a string as the sequence is deprecated and will raise a TypeError in future. It has been converted to a Seq object.
  warnings.warn(
2025-05-27 18:52:16,538 - INFO - Processing 5L3P_1_x.pdb
/Users/xiaojuzhang/opt/anaconda3/lib/python3.9/site-packages/Bio/SeqRecord.py:228: BiopythonDeprecationWarning: Using a string as the sequence is 

In [ ]:
#!python rna_loop_extractor.py --input-dir reconstructed_pdbs --output-dir extracted_loops --distance-cutoff 10.0

## D. RNA Fragment Extraction

### Extracting random RNA segments

In [1]:
from utils.extract_rna_segments import RNAExtractor

# Example 1: Extract segments with default coverage rate
extractor = RNAExtractor(
    input_dir="reconstructed_pdbs_gt30",
    min_length=5,
    max_length=20,
    coverage_rate=0.0002,
    generation_id="run3",
    output_dir="extracted_rna_segments_run3"  # Specify exact output location
)

# Process all files
extractor.process()

# # Example 2: Extract segments of different lengths with different coverage rates
# length_ranges = [
#     (5, 10, 0.15),   # 15% coverage for short segments
#     (10, 15, 0.1),   # 10% coverage for medium segments
#     (15, 20, 0.05)   # 5% coverage for longer segments
# ]

# for min_len, max_len, coverage in length_ranges:
#     extractor = RNAExtractor(
#         input_dir="processed_pdbs",
#         min_length=min_len,
#         max_length=max_len,
#         coverage_rate=coverage
#     )
#     extractor.process()

2025-06-15 19:39:44,132 - INFO - Found 1456 PDB files to process
2025-06-15 19:39:44,138 - INFO - Processing segments with min_length=3, max_length=20
2025-06-15 19:39:44,138 - INFO - Input segments: [(1, 71)]
2025-06-15 19:39:44,138 - INFO - Generated 1089 possible segments
2025-06-15 19:39:44,139 - INFO - Selected 1 segments for extraction
2025-06-15 19:39:44,141 - INFO - Successfully extracted 1 segments
/Users/xiaojuzhang/opt/anaconda3/lib/python3.9/site-packages/Bio/SeqRecord.py:228: BiopythonDeprecationWarning: Using a string as the sequence is deprecated and will raise a TypeError in future. It has been converted to a Seq object.
  warnings.warn(
2025-06-15 19:39:44,162 - INFO - Extracted segment from 9D0J chain A: 38-57 (length=20)
2025-06-15 19:39:44,309 - INFO - Processing segments with min_length=3, max_length=20
2025-06-15 19:39:44,309 - INFO - Input segments: [(1, 1750)]
2025-06-15 19:39:44,316 - INFO - Generated 31311 possible segments
2025-06-15 19:39:44,317 - INFO - Sel

KeyboardInterrupt: 

In [3]:
!python utils/extract_rna_segments.py --input-dir reconstructed_pdbs_gt30 --generation-id run3 --min-length 5 --max-length 20 --coverage-rate 0.0002        

2025-06-15 19:52:56,505 - INFO - Found 1456 PDB files to process
2025-06-15 19:52:56,510 - INFO - Processing segments with min_length=5, max_length=20
2025-06-15 19:52:56,510 - INFO - Input segments: [(1, 71)]
2025-06-15 19:52:56,510 - INFO - Generated 952 possible segments
2025-06-15 19:52:56,510 - INFO - Selected 0 segments for extraction
2025-06-15 19:52:56,510 - INFO - Successfully extracted 0 segments
2025-06-15 19:52:56,623 - INFO - Processing segments with min_length=5, max_length=20
2025-06-15 19:52:56,623 - INFO - Input segments: [(1, 1750)]
2025-06-15 19:52:56,628 - INFO - Generated 27816 possible segments
2025-06-15 19:52:56,629 - INFO - Selected 5 segments for extraction
2025-06-15 19:52:56,645 - INFO - Successfully extracted 5 segments
/Users/xiaojuzhang/opt/anaconda3/lib/python3.9/site-packages/Bio/SeqRecord.py:228: BiopythonDeprecationWarning: Using a string as the sequence is deprecated and will raise a TypeError in future. It has been converted to a Seq object.
  warni

## E. Convert PDB to NPY and extract sequences

## F. Sequence Analysis

### Extracting FASTA sequences

In [6]:
# Initialize the extractor
extractor = SequenceExtractor(
    pdb_dir="processed_pdbs",
    output_dir="sequences"
)

# Process all files
extractor.process()

# Example: Analyze sequence properties
def analyze_sequence(fasta_file):
    with open(fasta_file, 'r') as f:
        lines = f.readlines()
    
    sequence = ''.join(lines[1:]).strip()
    print(f"Sequence length: {len(sequence)}")
    print(f"Base composition: {dict(zip('AUGC', [sequence.count(b) for b in 'AUGC']))}")

# Analyze a specific file
fasta_file = "sequences/1ABC.fasta"
if os.path.exists(fasta_file):
    analyze_sequence(fasta_file)

NameError: name 'SequenceExtractor' is not defined

In [ ]:
!python scripts/pdb_to_fasta.py --input-dir reconstructed_pdbs --output-dir sequences

## F Construct official in-house test set
Filter the downloaded PDBs based on the match criters (merge file saved in merged_results_match_filtered.xlxs)

In [ ]:
! ython utils/copy_files.py --csv merged_results_match_filtered.csv --pdb_folder processed_pdbs --fasta_folder processed_rna_sequences --output_folder destination_folder

## Error Handling and Best Practices

Here are some examples of proper error handling and best practices:

In [ ]:
# Example 1: Safe file operations
def safe_file_operation(file_path):
    try:
        with open(file_path, 'r') as f:
            return f.read()
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return None
    except Exception as e:
        print(f"Error reading file: {e}")
        return None

# Example 2: Batch processing with progress tracking
def process_files(file_list):
    total = len(file_list)
    for i, file in enumerate(file_list, 1):
        print(f"Processing file {i}/{total}: {file}")
        try:
            # Process file
            pass
        except Exception as e:
            print(f"Error processing {file}: {e}")
            continue

# Example 3: Memory-efficient processing
def process_large_file(file_path, chunk_size=1000):
    with open(file_path, 'r') as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            # Process chunk
            yield chunk